# 1. 결산안 및 통장거래내역 형식 검사

## 패키지 import

In [17]:
# =========================
# 공통 import / 기본 설정
# =========================

import re
import calendar
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from difflib import SequenceMatcher

from openpyxl import load_workbook
from openpyxl.utils import get_column_letter

## 실행 정보 / 파일 경로 입력

In [18]:
# 샘플 파일 경로
settlement_file = r"C:\Users\daeha\Projects\student_audit_automation_system\data\private\2025학년도_03월_음악과_학생회_결산안.xlsx"
bank_file = r"C:\Users\daeha\Projects\student_audit_automation_system\data\private\2025학년도_03월_음악과_학생회_통장거래내역서_1차.xlsx"

# 실행 정보
audit_round = "1차"
org_name = "음악과 학생회"
target_month = "2025-03"
homepage_uploaded_at = "2025-04-05 22:30"

## 상수 / 공통 함수

In [19]:
STANDARD_EVIDENCE_LABELS = {
    "입금증빙", "정상영수증", "카드전표", "거래명세표", "이체확인증",
    "전월이월금", "현금수령증", "단체현금수령증", "간이영수증",
    "상금수령증", "상품수령증", "고액상품수령증", "사유서",
    "전기이월금", "식대사용보고안", "추가증빙자료"
}


def safe_load_workbook(file_path, data_only=False):
    try:
        wb = load_workbook(file_path, data_only=data_only)
        return wb, None
    except Exception as e:
        return None, str(e)

from datetime import datetime
import pandas as pd
import re

def try_parse_datetime(val):
    if pd.isna(val):
        return None

    if isinstance(val, datetime):
        return val

    text = str(val).strip()

    # 결산안 날짜 형식: '25.03.17'
    try:
        return datetime.strptime(text, "%y.%m.%d")
    except:
        return None


def try_parse_settlement_date(val, target_month=None):
    dt = try_parse_datetime(val)
    if dt is None:
        return None

    # target_month 예: '2025-03'
    if target_month:
        y, m = map(int, target_month.split("-"))
        if dt.year != y or dt.month != m:
            return None

    return dt

## 결산안 기본 구조 탐색 함수

In [20]:
STANDARD_EVIDENCE_LABELS = {
    "입금증빙", "정상영수증", "카드전표", "거래명세표", "이체확인증",
    "전월이월금", "현금수령증", "단체현금수령증", "간이영수증",
    "상금수령증", "상품수령증", "고액상품수령증", "사유서",
    "전기이월금", "식대사용보고안", "추가증빙자료"
}


def safe_load_workbook(file_path, data_only=False):
    try:
        wb = load_workbook(file_path, data_only=data_only)
        return wb, None
    except Exception as e:
        return None, str(e)

from datetime import datetime
import pandas as pd
import re

def try_parse_datetime(val):
    if pd.isna(val):
        return None

    if isinstance(val, datetime):
        return val

    text = str(val).strip()

    # 결산안 날짜 형식: '25.03.17'
    try:
        return datetime.strptime(text, "%y.%m.%d")
    except:
        return None


def try_parse_settlement_date(val, target_month=None):
    dt = try_parse_datetime(val)
    if dt is None:
        return None

    # target_month 예: '2025-03'
    if target_month:
        y, m = map(int, target_month.split("-"))
        if dt.year != y or dt.month != m:
            return None

    return dt

## 파일명 / 시트명 / 시트 보호 검사

In [21]:
def detect_header_row(ws):
    for row in ws.iter_rows(min_row=1, max_row=20):
        values = [cell.value for cell in row]
        values_str = [str(v).strip() if v is not None else "" for v in values]

        if "날짜" in values_str and "거래처명" in values_str:
            return row[0].row
    return None


def get_column_index_map(ws, header_row):
    headers = [ws.cell(header_row, col).value for col in range(1, ws.max_column + 1)]
    return {str(v).strip(): i + 1 for i, v in enumerate(headers) if v is not None}


def is_blank(value):
    return value is None or str(value).strip() == ""

## 결산안 수식 검사

In [22]:
def check_formula_usage(ws):
    """
    결산안 잔액 수식 검사
    - 데이터 행마다 잔액 셀에 수식이 있어야 함
    - 첫 데이터 행: 같은 행의 수입/지출을 참조해야 함
    - 이후 행: 직전 행 잔액 + 현재 행 수입 - 현재 행 지출 구조여야 함
    """
    header_row = detect_header_row(ws)
    if header_row is None:
        return {
            "rule_id": "RULE-FMT-005",
            "rule_name": "수식 사용 여부",
            "result_status": "REVIEW",
            "issue_summary": "헤더 행을 찾지 못해 수식 검사가 어렵습니다.",
            "expected_fix": "결산안 시트 구조를 확인하세요.",
            "predicted_penalty_points": 0
        }

    col_map = get_column_index_map(ws, header_row)

    required_cols = ["날짜", "거래처명", "품목", "수입", "지출", "잔액"]
    missing = [c for c in required_cols if c not in col_map]
    if missing:
        return {
            "rule_id": "RULE-FMT-005",
            "rule_name": "수식 사용 여부",
            "result_status": "REVIEW",
            "issue_summary": f"필수 열을 찾지 못했습니다: {missing}",
            "expected_fix": "결산안 열 구조를 확인하세요.",
            "predicted_penalty_points": 0
        }

    income_col_idx = col_map["수입"]
    expense_col_idx = col_map["지출"]
    balance_col_idx = col_map["잔액"]

    income_col_letter = get_column_letter(income_col_idx)
    expense_col_letter = get_column_letter(expense_col_idx)
    balance_col_letter = get_column_letter(balance_col_idx)

    def is_data_row(row_idx):
        values = {
            "날짜": ws.cell(row_idx, col_map["날짜"]).value,
            "거래처명": ws.cell(row_idx, col_map["거래처명"]).value,
            "품목": ws.cell(row_idx, col_map["품목"]).value,
            "수입": ws.cell(row_idx, income_col_idx).value,
            "지출": ws.cell(row_idx, expense_col_idx).value,
            "잔액": ws.cell(row_idx, balance_col_idx).value,
        }
        return any(v is not None and str(v).strip() != "" for v in values.values())

    data_rows = [row_idx for row_idx in range(header_row + 1, ws.max_row + 1) if is_data_row(row_idx)]

    if not data_rows:
        return {
            "rule_id": "RULE-FMT-005",
            "rule_name": "수식 사용 여부",
            "result_status": "REVIEW",
            "issue_summary": "데이터 행을 찾지 못했습니다.",
            "expected_fix": "결산안 데이터 영역을 확인하세요.",
            "predicted_penalty_points": 0
        }

    missing_formula_rows = []
    invalid_formula_rows = []

    for i, row_idx in enumerate(data_rows):
        balance_cell = ws.cell(row_idx, balance_col_idx)
        formula = balance_cell.value

        if not (isinstance(formula, str) and formula.startswith("=")):
            missing_formula_rows.append(row_idx)
            continue

        formula_upper = formula.upper().replace("$", "").replace(" ", "")

        has_current_income = f"{income_col_letter}{row_idx}" in formula_upper
        has_current_expense = f"{expense_col_letter}{row_idx}" in formula_upper

        if i == 0:
            if not (has_current_income and has_current_expense):
                invalid_formula_rows.append((row_idx, formula))
        else:
            prev_row_idx = data_rows[i - 1]
            has_prev_balance = f"{balance_col_letter}{prev_row_idx}" in formula_upper

            if not (has_prev_balance and has_current_income and has_current_expense):
                invalid_formula_rows.append((row_idx, formula))

    if missing_formula_rows or invalid_formula_rows:
        msgs = []
        if missing_formula_rows:
            if len(missing_formula_rows) <= 10:
                msgs.append(f"잔액 수식 누락 행: {missing_formula_rows}")
            else:
                msgs.append(f"잔액 수식 누락 행 다수 존재 (예: {missing_formula_rows[:10]} ...)")

        if invalid_formula_rows:
            sample = [f"{r}행({f})" for r, f in invalid_formula_rows[:5]]
            msgs.append(f"잔액 수식 구조 이상 행: {sample}")

        return {
            "rule_id": "RULE-FMT-005",
            "rule_name": "수식 사용 여부",
            "result_status": "FAIL",
            "issue_summary": " / ".join(msgs),
            "expected_fix": "모든 데이터 행의 잔액 셀에 엑셀 수식을 적용하고, 첫 행 이후에는 직전 행 잔액을 참조하도록 통일하세요.",
            "predicted_penalty_points": 2
        }

    return {
        "rule_id": "RULE-FMT-005",
        "rule_name": "수식 사용 여부",
        "result_status": "PASS",
        "issue_summary": None,
        "expected_fix": None,
        "predicted_penalty_points": 0
    }

## 결산안 데이터 추출 / 증빙자료명 검사

In [23]:
def check_formula_usage(ws):
    """
    결산안 잔액 수식 검사
    - 데이터 행마다 잔액 셀에 수식이 있어야 함
    - 첫 데이터 행: 같은 행의 수입/지출을 참조해야 함
    - 이후 행: 직전 행 잔액 + 현재 행 수입 - 현재 행 지출 구조여야 함
    """
    header_row = detect_header_row(ws)
    if header_row is None:
        return {
            "rule_id": "RULE-FMT-005",
            "rule_name": "수식 사용 여부",
            "result_status": "REVIEW",
            "issue_summary": "헤더 행을 찾지 못해 수식 검사가 어렵습니다.",
            "expected_fix": "결산안 시트 구조를 확인하세요.",
            "predicted_penalty_points": 0
        }

    col_map = get_column_index_map(ws, header_row)

    required_cols = ["날짜", "거래처명", "품목", "수입", "지출", "잔액"]
    missing = [c for c in required_cols if c not in col_map]
    if missing:
        return {
            "rule_id": "RULE-FMT-005",
            "rule_name": "수식 사용 여부",
            "result_status": "REVIEW",
            "issue_summary": f"필수 열을 찾지 못했습니다: {missing}",
            "expected_fix": "결산안 열 구조를 확인하세요.",
            "predicted_penalty_points": 0
        }

    income_col_idx = col_map["수입"]
    expense_col_idx = col_map["지출"]
    balance_col_idx = col_map["잔액"]

    income_col_letter = get_column_letter(income_col_idx)
    expense_col_letter = get_column_letter(expense_col_idx)
    balance_col_letter = get_column_letter(balance_col_idx)

    def is_data_row(row_idx):
        values = {
            "날짜": ws.cell(row_idx, col_map["날짜"]).value,
            "거래처명": ws.cell(row_idx, col_map["거래처명"]).value,
            "품목": ws.cell(row_idx, col_map["품목"]).value,
            "수입": ws.cell(row_idx, income_col_idx).value,
            "지출": ws.cell(row_idx, expense_col_idx).value,
            "잔액": ws.cell(row_idx, balance_col_idx).value,
        }
        return any(v is not None and str(v).strip() != "" for v in values.values())

    data_rows = [row_idx for row_idx in range(header_row + 1, ws.max_row + 1) if is_data_row(row_idx)]

    if not data_rows:
        return {
            "rule_id": "RULE-FMT-005",
            "rule_name": "수식 사용 여부",
            "result_status": "REVIEW",
            "issue_summary": "데이터 행을 찾지 못했습니다.",
            "expected_fix": "결산안 데이터 영역을 확인하세요.",
            "predicted_penalty_points": 0
        }

    missing_formula_rows = []
    invalid_formula_rows = []

    for i, row_idx in enumerate(data_rows):
        balance_cell = ws.cell(row_idx, balance_col_idx)
        formula = balance_cell.value

        if not (isinstance(formula, str) and formula.startswith("=")):
            missing_formula_rows.append(row_idx)
            continue

        formula_upper = formula.upper().replace("$", "").replace(" ", "")

        has_current_income = f"{income_col_letter}{row_idx}" in formula_upper
        has_current_expense = f"{expense_col_letter}{row_idx}" in formula_upper

        if i == 0:
            if not (has_current_income and has_current_expense):
                invalid_formula_rows.append((row_idx, formula))
        else:
            prev_row_idx = data_rows[i - 1]
            has_prev_balance = f"{balance_col_letter}{prev_row_idx}" in formula_upper

            if not (has_prev_balance and has_current_income and has_current_expense):
                invalid_formula_rows.append((row_idx, formula))

    if missing_formula_rows or invalid_formula_rows:
        msgs = []
        if missing_formula_rows:
            if len(missing_formula_rows) <= 10:
                msgs.append(f"잔액 수식 누락 행: {missing_formula_rows}")
            else:
                msgs.append(f"잔액 수식 누락 행 다수 존재 (예: {missing_formula_rows[:10]} ...)")

        if invalid_formula_rows:
            sample = [f"{r}행({f})" for r, f in invalid_formula_rows[:5]]
            msgs.append(f"잔액 수식 구조 이상 행: {sample}")

        return {
            "rule_id": "RULE-FMT-005",
            "rule_name": "수식 사용 여부",
            "result_status": "FAIL",
            "issue_summary": " / ".join(msgs),
            "expected_fix": "모든 데이터 행의 잔액 셀에 엑셀 수식을 적용하고, 첫 행 이후에는 직전 행 잔액을 참조하도록 통일하세요.",
            "predicted_penalty_points": 2
        }

    return {
        "rule_id": "RULE-FMT-005",
        "rule_name": "수식 사용 여부",
        "result_status": "PASS",
        "issue_summary": None,
        "expected_fix": None,
        "predicted_penalty_points": 0
    }

## 통장 조회기간 / 정렬 검사

In [24]:
def extract_settlement_rows(ws):
    header_row = detect_header_row(ws)
    if header_row is None:
        return pd.DataFrame()

    col_map = get_column_index_map(ws, header_row)
    rows = []

    for row_idx in range(header_row + 1, ws.max_row + 1):
        date_val = ws.cell(row_idx, col_map.get("날짜", 1)).value if "날짜" in col_map else None
        vendor_val = ws.cell(row_idx, col_map.get("거래처명", 3)).value if "거래처명" in col_map else None
        item_val = ws.cell(row_idx, col_map.get("품목", 4)).value if "품목" in col_map else None

        if date_val is None and vendor_val is None and item_val is None:
            continue

        row_data = {"row_index": row_idx}
        for key, col_idx in col_map.items():
            row_data[key] = ws.cell(row_idx, col_idx).value
        rows.append(row_data)

    return pd.DataFrame(rows)


def check_evidence_label_standard(df):
    if df.empty or "증빙자료" not in df.columns:
        evidence_cols = [c for c in df.columns if "증빙" in str(c)]
        if not evidence_cols:
            return [{
                "rule_id": "RULE-FMT-008",
                "rule_name": "증빙자료명 표준화",
                "result_status": "REVIEW",
                "issue_summary": "증빙자료 관련 열을 찾지 못했습니다.",
                "expected_fix": "결산안 열 구조를 확인하세요.",
                "predicted_penalty_points": 0
            }]
        col_name = evidence_cols[0]
    else:
        col_name = "증빙자료"

    results = []

    for _, row in df.iterrows():
        value = row.get(col_name)
        if pd.isna(value) or value is None or str(value).strip() == "":
            continue

        labels = [x.strip() for x in str(value).split(",")]
        invalid_labels = [x for x in labels if x not in STANDARD_EVIDENCE_LABELS]

        if invalid_labels:
            results.append({
                "rule_id": "RULE-FMT-008",
                "rule_name": "증빙자료명 표준화",
                "result_status": "FAIL",
                "issue_summary": f"{int(row['row_index'])}행 증빙자료명 비표준 사용: {invalid_labels}",
                "expected_fix": "표준 증빙자료명으로 수정하세요.",
                "predicted_penalty_points": 2
            })

    if not results:
        results.append({
            "rule_id": "RULE-FMT-008",
            "rule_name": "증빙자료명 표준화",
            "result_status": "PASS",
            "issue_summary": None,
            "expected_fix": None,
            "predicted_penalty_points": 0
        })

    return results

## 셀합치기 / 불필요한 행 / 결산안 정렬 / 제목 연월

In [25]:
def find_next_non_empty_in_row(ws, row_idx, start_col_idx):
    for col_idx in range(start_col_idx + 1, ws.max_column + 1):
        val = ws.cell(row_idx, col_idx).value
        if val is not None and str(val).strip() != "":
            return val
    return None


def extract_bank_query_period(ws):
    for row_idx in range(1, min(ws.max_row, 15) + 1):
        for col_idx in range(1, ws.max_column + 1):
            val = ws.cell(row_idx, col_idx).value
            if val is not None and str(val).strip() == "조회기간":
                period_val = find_next_non_empty_in_row(ws, row_idx, col_idx)
                if period_val is not None:
                    return str(period_val).strip()
    return None


def check_bank_query_period(ws, target_month):
    period_text = extract_bank_query_period(ws)

    if not period_text:
        return {
            "rule_id": "RULE-FMT-009",
            "rule_name": "통장 조회기간",
            "result_status": "REVIEW",
            "issue_summary": "통장 시트에서 조회기간을 찾지 못했습니다.",
            "expected_fix": "통장거래내역 상단의 조회기간을 확인하세요.",
            "predicted_penalty_points": 0
        }

    m = re.search(r"(\d{4})[.\-](\d{2})[.\-](\d{2})\s*[-~]\s*(\d{4})[.\-](\d{2})[.\-](\d{2})", period_text)
    if not m:
        return {
            "rule_id": "RULE-FMT-009",
            "rule_name": "통장 조회기간",
            "result_status": "REVIEW",
            "issue_summary": f"조회기간 형식을 해석하지 못했습니다. 현재 값: {period_text}",
            "expected_fix": "조회기간 형식을 확인하세요.",
            "predicted_penalty_points": 0
        }

    start_y, start_m, start_d, end_y, end_m, end_d = map(int, m.groups())

    year, month = map(int, target_month.split("-"))
    last_day = calendar.monthrange(year, month)[1]

    expected_start = (year, month, 1)
    expected_end = (year, month, last_day)
    actual_start = (start_y, start_m, start_d)
    actual_end = (end_y, end_m, end_d)

    passed = (actual_start == expected_start) and (actual_end == expected_end)

    return {
        "rule_id": "RULE-FMT-009",
        "rule_name": "통장 조회기간",
        "result_status": "PASS" if passed else "FAIL",
        "issue_summary": None if passed else f"조회기간이 당월 1일~말일이 아닙니다. 현재: {period_text}",
        "expected_fix": None if passed else f"조회기간을 {year}.{month:02d}.01 - {year}.{month:02d}.{last_day:02d} 로 설정하세요.",
        "predicted_penalty_points": 2 if not passed else 0
    }


def find_bank_header_row(ws):
    for row in ws.iter_rows(min_row=1, max_row=20):
        values = [cell.value for cell in row]
        values_str = [str(v).strip() if v is not None else "" for v in values]

        if "거래일시" in values_str and "구분" in values_str and "거래금액" in values_str:
            return row[0].row
    return None


def try_parse_bank_datetime(val):
    if pd.isna(val):
        return None

    if isinstance(val, datetime):
        return val

    text = str(val).strip()

    patterns = [
        "%Y.%m.%d %H:%M:%S",
        "%Y-%m-%d %H:%M:%S",
        "%Y/%m/%d %H:%M:%S",
        "%Y.%m.%d %H:%M",
        "%Y-%m-%d %H:%M",
        "%Y/%m/%d %H:%M",
        "%Y.%m.%d",
        "%Y-%m-%d",
        "%Y/%m/%d",
    ]

    for fmt in patterns:
        try:
            return datetime.strptime(text, fmt)
        except:
            pass

    return None


def extract_bank_table_datetimes(ws):
    header_row = find_bank_header_row(ws)
    if header_row is None:
        return []

    headers = [ws.cell(header_row, col).value for col in range(1, ws.max_column + 1)]
    col_map = {str(v).strip(): i + 1 for i, v in enumerate(headers) if v is not None}

    if "거래일시" not in col_map:
        return []

    dt_col = col_map["거래일시"]
    dts = []

    for row_idx in range(header_row + 1, ws.max_row + 1):
        val = ws.cell(row_idx, dt_col).value
        dt = try_parse_bank_datetime(val)
        if dt:
            dts.append(dt)

    return dts


def check_bank_row_order(ws):
    dts = extract_bank_table_datetimes(ws)

    if len(dts) < 2:
        return {
            "rule_id": "RULE-FMT-006",
            "rule_name": "통장 정렬 방향",
            "result_status": "REVIEW",
            "issue_summary": "거래 테이블의 거래일시를 충분히 찾지 못했습니다.",
            "expected_fix": "통장거래내역 형식을 확인하세요.",
            "predicted_penalty_points": 0
        }

    ascending = all(dts[i] <= dts[i + 1] for i in range(len(dts) - 1))

    return {
        "rule_id": "RULE-FMT-006",
        "rule_name": "통장 정렬 방향",
        "result_status": "PASS" if ascending else "FAIL",
        "issue_summary": None if ascending else "통장거래내역이 과거→최신 순으로 정렬되어 있지 않습니다.",
        "expected_fix": None if ascending else "과거 거래가 위로 오도록 저장하세요.",
        "predicted_penalty_points": 1 if not ascending else 0
    }

## 제출기한 검사

In [26]:
def check_merged_cells_limit(ws, limit=10):
    merged_count = len(ws.merged_cells.ranges)
    passed = merged_count <= limit

    return {
        "rule_id": "RULE-FMT-010",
        "rule_name": "결산안 셀 합치기 개수",
        "result_status": "PASS" if passed else "FAIL",
        "issue_summary": None if passed else f"결산안 셀 합치기가 {merged_count}개로 {limit}개를 초과했습니다.",
        "expected_fix": None if passed else f"셀 합치기 개수를 {limit}개 이하로 줄이세요.",
        "predicted_penalty_points": 2 if not passed else 0
    }


def check_unnecessary_blank_rows(ws):
    header_row = detect_header_row(ws)
    if header_row is None:
        return {
            "rule_id": "RULE-FMT-011",
            "rule_name": "결산안 불필요한 행",
            "result_status": "REVIEW",
            "issue_summary": "헤더 행을 찾지 못해 빈 행 검사가 어렵습니다.",
            "expected_fix": "결산안 시트 구조를 확인하세요.",
            "predicted_penalty_points": 0
        }

    col_map = get_column_index_map(ws, header_row)
    required_cols = ["날짜", "거래처명", "품목", "수입", "지출", "잔액"]
    available_cols = [c for c in required_cols if c in col_map]

    blank_rows = []
    started = False

    for row_idx in range(header_row + 1, ws.max_row + 1):
        values = [ws.cell(row_idx, col_map[c]).value for c in available_cols]
        row_is_blank = all(is_blank(v) for v in values)

        if not row_is_blank:
            started = True
            continue

        if started:
            has_data_below = False
            for next_row in range(row_idx + 1, ws.max_row + 1):
                next_values = [ws.cell(next_row, col_map[c]).value for c in available_cols]
                if any(not is_blank(v) for v in next_values):
                    has_data_below = True
                    break

            if has_data_below:
                blank_rows.append(row_idx)

    return {
        "rule_id": "RULE-FMT-011",
        "rule_name": "결산안 불필요한 행",
        "result_status": "PASS" if not blank_rows else "FAIL",
        "issue_summary": None if not blank_rows else f"중간 빈 행이 존재합니다: {blank_rows[:10]}",
        "expected_fix": None if not blank_rows else "결산안 중간의 불필요한 빈 행을 삭제하세요.",
        "predicted_penalty_points": 2 if blank_rows else 0
    }

def extract_settlement_date_sequence(ws, target_month):
    header_row = detect_header_row(ws)
    if header_row is None:
        return [], []

    col_map = get_column_index_map(ws, header_row)
    if "날짜" not in col_map:
        return [], []

    date_col = col_map["날짜"]
    rows = []
    failed_values = []

    for row_idx in range(header_row + 1, ws.max_row + 1):
        raw_val = ws.cell(row_idx, date_col).value

        if is_blank(raw_val):
            continue

        dt = try_parse_settlement_date(raw_val, target_month)
        if dt is not None:
            rows.append((row_idx, dt))
        else:
            failed_values.append((row_idx, raw_val))

    return rows, failed_values


def check_settlement_row_order(ws, target_month):
    date_rows, failed_values = extract_settlement_date_sequence(ws, target_month)

    if len(date_rows) < 2:
        return {
            "rule_id": "RULE-FMT-007",
            "rule_name": "결산안 정렬 방향",
            "result_status": "REVIEW",
            "issue_summary": f"결산안 날짜를 충분히 찾지 못했습니다. 파싱 실패 예시: {failed_values[:5]}",
            "expected_fix": "결산안 날짜 열 형식을 확인하세요.",
            "predicted_penalty_points": 0
        }

    dts = [dt for _, dt in date_rows]
    ascending = all(dts[i] <= dts[i + 1] for i in range(len(dts) - 1))

    return {
        "rule_id": "RULE-FMT-007",
        "rule_name": "결산안 정렬 방향",
        "result_status": "PASS" if ascending else "FAIL",
        "issue_summary": None if ascending else "결산안이 과거→최신 순으로 정렬되어 있지 않습니다.",
        "expected_fix": None if ascending else "결산안도 과거 거래가 위로 오도록 정렬하세요.",
        "predicted_penalty_points": 1 if not ascending else 0
    }


def check_settlement_title_month_consistency(ws, target_month):
    target_year, target_month_num = map(int, target_month.split("-"))

    # 1~2행만 확인
    text_pool = []
    for row_idx in range(1, 3):
        for col_idx in range(1, ws.max_column + 1):
            val = ws.cell(row_idx, col_idx).value
            if val is not None and str(val).strip() != "":
                text_pool.append(str(val).strip())

    joined_text = " ".join(text_pool)

    # 허용 형식 1: 2025년 월별 사무결산
    # 허용 형식 2: 2025년 1월 사무결산
    if re.search(rf"{target_year}\s*년\s*월별\s*사무결산", joined_text):
        return {
            "rule_id": "RULE-CNS-001",
            "rule_name": "결산안 상단 제목 연월 일치",
            "result_status": "PASS",
            "issue_summary": None,
            "expected_fix": None,
            "predicted_penalty_points": 0
        }

    if re.search(rf"{target_year}\s*년\s*{target_month_num}\s*월\s*사무결산", joined_text):
        return {
            "rule_id": "RULE-CNS-001",
            "rule_name": "결산안 상단 제목 연월 일치",
            "result_status": "PASS",
            "issue_summary": None,
            "expected_fix": None,
            "predicted_penalty_points": 0
        }

    return {
        "rule_id": "RULE-CNS-001",
        "rule_name": "결산안 상단 제목 연월 일치",
        "result_status": "REVIEW",
        "issue_summary": f"상단 제목에서 연도/월 정보를 찾지 못했습니다. 제목 내용: {joined_text}",
        "expected_fix": "상단 제목이 '2025년 월별 사무결산' 또는 '2025년 3월 사무결산' 형식인지 확인하세요.",
        "predicted_penalty_points": 0
    }

## 전체 실행

In [27]:
results = []

settlement_name = Path(settlement_file).name
bank_name = Path(bank_file).name

# =========================
# 1. 파일명 검사 (inline)
# =========================
settlement_pattern = r"^\d{4}학년도_\d{2}월_.+_학생회_결산안(_[123]차)?\.xlsx$"
settlement_passed = bool(re.match(settlement_pattern, settlement_name))

results.append({
    "rule_id": "RULE-FMT-001",
    "rule_name": "결산안 파일명 형식",
    "result_status": "PASS" if settlement_passed else "FAIL",
    "issue_summary": None if settlement_passed else "결산안 파일명에 차수(1차/2차/3차)가 없거나 형식이 다릅니다.",
    "expected_fix": None if settlement_passed else "파일명을 '2025학년도_03월_음악과_학생회_결산안_1차.xlsx' 형식으로 수정하세요.",
    "predicted_penalty_points": 0 if settlement_passed else 2
})

bank_pattern = r"^\d{4}학년도_\d{2}월_.+_학생회_통장거래내역(서)?(_[123]차)?\.xlsx$"
bank_passed = bool(re.match(bank_pattern, bank_name))

results.append({
    "rule_id": "RULE-FMT-002",
    "rule_name": "통장거래내역 파일명 형식",
    "result_status": "PASS" if bank_passed else "FAIL",
    "issue_summary": None if bank_passed else "통장거래내역 파일명 형식이 교육자료 기준과 다릅니다.",
    "expected_fix": None if bank_passed else "파일명을 '2025학년도_03월_음악과_학생회_통장거래내역_1차.xlsx' 형식으로 수정하세요.",
    "predicted_penalty_points": 0 if bank_passed else 2
})

# =========================
# 2. 파일 열기
# =========================
settlement_wb, settlement_err = safe_load_workbook(settlement_file, data_only=False)
bank_wb, bank_err = safe_load_workbook(bank_file, data_only=False)

# =========================
# 3. 결산안 검사
# =========================
if settlement_wb:
    ws = settlement_wb[settlement_wb.sheetnames[0]]

    # 시트명 검사 (inline)
    sheet_name_passed = (str(ws.title).strip() == "결산안")
    results.append({
        "rule_id": "RULE-FMT-003",
        "rule_name": "결산안 시트명",
        "result_status": "PASS" if sheet_name_passed else "FAIL",
        "issue_summary": None if sheet_name_passed else f"결산안 시트명이 '결산안'이 아닙니다. 현재 시트명: {ws.title}",
        "expected_fix": None if sheet_name_passed else "시트명을 '결산안'으로 수정하세요.",
        "predicted_penalty_points": 0 if sheet_name_passed else 1
    })

    # 시트 보호 검사 (inline)
    sheet_protected = bool(ws.protection.sheet)
    results.append({
        "rule_id": "RULE-FMT-004",
        "rule_name": "결산안 시트 보호",
        "result_status": "PASS" if sheet_protected else "FAIL",
        "issue_summary": None if sheet_protected else "결산안 시트 보호가 적용되어 있지 않습니다.",
        "expected_fix": None if sheet_protected else "시트 보호를 적용하세요.",
        "predicted_penalty_points": 0 if sheet_protected else 2
    })

    # 기존 정의 함수 사용
    results.append(check_formula_usage(ws))
    results.append(check_merged_cells_limit(ws))
    results.append(check_unnecessary_blank_rows(ws))
    results.append(check_settlement_row_order(ws, target_month))
    results.append(check_settlement_title_month_consistency(ws, target_month))

    settlement_df = extract_settlement_rows(ws)
    results.extend(check_evidence_label_standard(settlement_df))

else:
    results.append({
        "rule_id": "SETTLEMENT-LOAD",
        "rule_name": "결산안 파일 열기",
        "result_status": "BLOCKED",
        "issue_summary": settlement_err,
        "expected_fix": "파일 상태를 확인하세요.",
        "predicted_penalty_points": 0
    })

# =========================
# 4. 통장 검사
# =========================
if bank_wb:
    bank_ws = bank_wb[bank_wb.sheetnames[0]]

    results.append(check_bank_row_order(bank_ws))
    results.append(check_bank_query_period(bank_ws, target_month))

else:
    results.append({
        "rule_id": "BANK-LOAD",
        "rule_name": "통장 파일 열기",
        "result_status": "BLOCKED",
        "issue_summary": bank_err,
        "expected_fix": "암호 해제 또는 파일 상태 확인이 필요합니다.",
        "predicted_penalty_points": 2
    })

# =========================
# 5. 제출기한 검사 (inline)
# =========================
uploaded_dt = pd.to_datetime(homepage_uploaded_at)
deadline = pd.Timestamp("2025-04-05 23:59:59")
deadline_passed = uploaded_dt <= deadline

results.append({
    "rule_id": "RULE-PRC-001",
    "rule_name": "제출기한 준수 여부",
    "result_status": "PASS" if deadline_passed else "FAIL",
    "issue_summary": "기한 내 제출" if deadline_passed else f"제출기한 초과 ({uploaded_dt})",
    "expected_fix": None if deadline_passed else "마감 기한 내 업로드가 필요합니다.",
    "predicted_penalty_points": 0
})

result_df = pd.DataFrame(results)
result_df

c:\Users\daeha\AppData\Local\Programs\Python\Python314\Lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


,rule_id,rule_name,result_status,issue_summary,expected_fix,predicted_penalty_points
0,RULE-FMT-001,결산안 파일명 형식,PASS,NaN,NaN,0
1,RULE-FMT-002,통장거래내역 파일명 형식,PASS,NaN,NaN,0
2,RULE-FMT-003,결산안 시트명,FAIL,결산안 시트명이 '결산안'이 아닙니다. 현재 시트명: 03월 결산안,시트명을 '결산안'으로 수정하세요.,1
3,RULE-FMT-004,결산안 시트 보호,FAIL,결산안 시트 보호가 적용되어 있지 않습니다.,시트 보호를 적용하세요.,2
4,RULE-FMT-005,수식 사용 여부,FAIL,잔액 수식 누락 행: [4] / 잔액 수식 구조 이상 행: ['112행(=E112-...,"모든 데이터 행의 잔액 셀에 엑셀 수식을 적용하고, 첫 행 이후에는 직전 행 잔액을...",2
5,RULE-FMT-010,결산안 셀 합치기 개수,PASS,NaN,NaN,0
6,RULE-FMT-011,결산안 불필요한 행,PASS,NaN,NaN,0
7,RULE-FMT-007,결산안 정렬 방향,PASS,NaN,NaN,0
8,RULE-CNS-001,결산안 상단 제목 연월 일치,PASS,NaN,NaN,0
9,RULE-FMT-008,증빙자료명 표준화,PASS,NaN,NaN,0


# 2. 결산안 및 통장거래내역 거래 매칭

In [43]:
# =========================
# 벌점 규칙 확립 + 거래 매칭 + 행별 감사
# (기존 거래 매칭 파트를 이 셀로 대체)
# =========================

import re
import calendar
from pathlib import Path
from datetime import datetime
from difflib import SequenceMatcher

import numpy as np
import pandas as pd
from openpyxl import load_workbook

# ---------------------------------
# 0. 규칙 DB
# ---------------------------------
PENALTY_RULES = {
    "양식 미준수": 2,
    "복구된 일시적 오사용/오적립": 2,
    "증빙자료 누락": 5,
    "기한 미준수(1~3일)": 5,
    "기한 미준수(4일 이상~익월5일)": 10,
    "미제출": 20,
    "학생회비 오용": 20,
    "검토 필요": 0,
}

STANDARD_EVIDENCE_LABELS = {
    "입금증빙", "정상영수증", "카드전표", "거래명세표", "이체확인증",
    "전월이월금", "현금수령증", "단체현금수령증", "간이영수증",
    "상금수령증", "상품수령증", "고액상품수령증", "사유서",
    "전기이월금", "식대사용보고안", "추가증빙자료"
}

# 교육자료상 비표준 표기 -> 표준 표기 추천용
EVIDENCE_ALIAS_MAP = {
    "현금영수증": "거래명세표",
    "영수증": "정상영수증",
    "주문내역": "추가증빙자료",
    "네이버결제": "거래명세표",
    "거래확인증": "정상영수증",
    "머니거래확인증": "추가증빙자료",
}

# 실제 데이터에서 많이 등장하는 개인 거래처 후보
PERSON_LIKE_VENDORS = {
    "김서현", "조효원", "문민영", "장예영", "장에영", "정이영", "트래블로버", "트래볼로버",
    "한연두", "최의선", "김혜진", "황다안", "이현지", "김민지", "박소현"
}

# ---------------------------------
# 1. 공통 함수
# ---------------------------------
def safe_load_workbook(file_path, data_only=False):
    try:
        return load_workbook(file_path, data_only=data_only), None
    except Exception as e:
        return None, str(e)

def is_blank(value):
    return value is None or str(value).strip() == ""

def clean_amount(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return 0
    if isinstance(value, (int, float, np.integer, np.floating)):
        return int(round(float(value)))
    text = str(value).strip().replace(",", "")
    if text == "":
        return 0
    try:
        return int(round(float(text)))
    except:
        return 0

def try_parse_settlement_date(val):
    if pd.isna(val):
        return None
    if isinstance(val, datetime):
        return val
    text = str(val).strip()
    for fmt in ("%y.%m.%d", "%Y.%m.%d", "%Y-%m-%d"):
        try:
            return datetime.strptime(text, fmt)
        except:
            pass
    return None

def try_parse_bank_datetime(val):
    if pd.isna(val):
        return None
    if isinstance(val, datetime):
        return val
    text = str(val).strip()
    for fmt in ("%Y.%m.%d %H:%M:%S", "%Y-%m-%d %H:%M:%S", "%Y.%m.%d"):
        try:
            return datetime.strptime(text, fmt)
        except:
            pass
    return None

def detect_settlement_header_row(ws):
    for row in ws.iter_rows(min_row=1, max_row=20):
        values = [str(c.value).strip() if c.value is not None else "" for c in row]
        if "날짜" in values and "거래처명" in values and "수입" in values and "지출" in values:
            return row[0].row
    return None

def get_column_index_map(ws, header_row):
    headers = [ws.cell(header_row, c).value for c in range(1, ws.max_column + 1)]
    return {str(v).strip(): i + 1 for i, v in enumerate(headers) if v is not None}

def find_bank_header_row(ws):
    for row_idx in range(1, 20):
        values = [str(ws.cell(row_idx, c).value).strip() if ws.cell(row_idx, c).value is not None else "" for c in range(1, ws.max_column + 1)]
        if "거래일시" in values and "거래금액" in values and "거래구분" in values:
            return row_idx
    return None

def split_label_number(label):
    label = str(label).strip()
    m = re.match(r"^(.*?)(\d+)$", label)
    if m:
        return m.group(1).strip(), int(m.group(2))
    return label, None

def normalize_evidence_label(label):
    if is_blank(label):
        return ""
    base, _ = split_label_number(label)
    return EVIDENCE_ALIAS_MAP.get(base, base)

def add_issue(rows, row_index, issue_type, issue_summary, expected_fix, row_snapshot=None):
    rows.append({
        "row_index": int(row_index) if row_index is not None else 0,
        "issue_type": issue_type,
        "penalty_points": PENALTY_RULES.get(issue_type, 0),
        "issue_summary": issue_summary,
        "expected_fix": expected_fix,
        "row_snapshot": row_snapshot
    })

# ---------------------------------
# 2. 형식 규칙
# ---------------------------------
def check_filename_rules(settlement_file, bank_file):
    issues = []

    settlement_name = Path(settlement_file).name
    bank_name = Path(bank_file).name

    settlement_pattern = r"^\d{4}학년도_\d{2}월_.+_학생회_결산안_[123]차\.xlsx$"
    bank_pattern = r"^\d{4}학년도_\d{2}월_.+_학생회_통장거래내역_[123]차\.xlsx$"

    if not re.match(settlement_pattern, settlement_name):
        add_issue(
            issues, 0, "양식 미준수",
            f"결산안 파일명 형식 미준수: {settlement_name}",
            "파일명을 '2025학년도_03월_음악과_학생회_결산안_1차.xlsx' 형식으로 수정하세요."
        )

    if not re.match(bank_pattern, bank_name):
        add_issue(
            issues, 0, "양식 미준수",
            f"통장거래내역 파일명 형식 미준수: {bank_name}",
            "파일명을 '2025학년도_03월_음악과_학생회_통장거래내역_1차.xlsx' 형식으로 수정하세요. ('통장거래내역서' 금지)"
        )

    return issues

def check_sheet_rules(ws, target_month):
    issues = []

    expected_title = f"{int(target_month.split('-')[1]):02d}월 결산안"
    if str(ws.title).strip() != expected_title:
        add_issue(
            issues, 0, "양식 미준수",
            f"결산안 시트명이 '{ws.title}'입니다.",
            f"시트명을 '{expected_title}'으로 수정하세요."
        )

    if not ws.protection.sheet:
        add_issue(
            issues, 0, "양식 미준수",
            "결산안 시트 보호가 되어 있지 않습니다.",
            "시트 보호를 적용하세요. (pw: sejong2025)"
        )

    merged_count = len(ws.merged_cells.ranges)
    if merged_count > 10:
        add_issue(
            issues, 0, "양식 미준수",
            f"셀 합치기 개수가 {merged_count}개로 10개를 초과했습니다.",
            "셀 합치기 개수를 10개 이하로 줄이세요."
        )

    # 제목 월 검사
    title_text = str(ws["A1"].value or "")
    m = re.search(r"(\d{4})년.*?(\d{1,2})월", title_text)
    if m:
        y, mth = int(m.group(1)), int(m.group(2))
        ty, tm = map(int, target_month.split("-"))
        if (y, mth) != (ty, tm):
            add_issue(
                issues, 0, "양식 미준수",
                f"A1 제목 연월이 target_month와 다릅니다. 현재: {y}-{mth:02d}",
                "제목 연월을 대상 월과 일치시키세요."
            )

    return issues

def check_bank_rules(ws, target_month):
    issues = []

    # 조회기간
    period_text = None
    for r in range(1, 15):
        for c in range(1, ws.max_column + 1):
            if str(ws.cell(r, c).value).strip() == "조회기간":
                for cc in range(c + 1, ws.max_column + 1):
                    val = ws.cell(r, cc).value
                    if not is_blank(val):
                        period_text = str(val).strip()
                        break

    if period_text:
        m = re.search(r"(\d{4})[.\-](\d{2})[.\-](\d{2})\s*[-~]\s*(\d{4})[.\-](\d{2})[.\-](\d{2})", period_text)
        if m:
            sy, sm, sd, ey, em, ed = map(int, m.groups())
            y, mth = map(int, target_month.split("-"))
            last_day = calendar.monthrange(y, mth)[1]

            if (sy, sm, sd) != (y, mth, 1) or (ey, em, ed) != (y, mth, last_day):
                add_issue(
                    issues, 0, "양식 미준수",
                    f"통장 조회기간 미준수: {period_text}",
                    f"조회기간을 {y}.{mth:02d}.01 - {y}.{mth:02d}.{last_day:02d} 로 다시 내려받으세요."
                )
        else:
            add_issue(
                issues, 0, "검토 필요",
                f"조회기간 형식을 해석하지 못했습니다: {period_text}",
                "통장 상단 조회기간 표기를 확인하세요."
            )

    # 과거 내역이 위로 오는지 검사
    header_row = find_bank_header_row(ws)
    if header_row is not None:
        datetimes = []
        for r in range(header_row + 1, ws.max_row + 1):
            dt = try_parse_bank_datetime(ws.cell(r, 2).value)
            if dt is not None:
                datetimes.append(dt)

        if datetimes:
            sorted_ok = all(datetimes[i] <= datetimes[i + 1] for i in range(len(datetimes) - 1))
            if not sorted_ok:
                add_issue(
                    issues, 0, "양식 미준수",
                    "통장거래내역 정렬이 교육자료 기준(과거 내역이 위)과 다릅니다.",
                    "통장거래내역을 오름차순으로 다시 저장하세요."
                )

    return issues

def check_due_date_rule(homepage_uploaded_at, target_month):
    issues = []
    if not homepage_uploaded_at:
        return issues

    uploaded_dt = pd.to_datetime(homepage_uploaded_at)
    year, month = map(int, target_month.split("-"))

    due_year = year
    due_month = month + 1
    if due_month == 13:
        due_year += 1
        due_month = 1

    due_dt = pd.Timestamp(due_year, due_month, 5, 23, 59, 59)

    if uploaded_dt > due_dt:
        delay_days = (uploaded_dt.date() - due_dt.date()).days
        issue_type = "기한 미준수(1~3일)" if delay_days <= 3 else "기한 미준수(4일 이상~익월5일)"
        add_issue(
            issues, 0, issue_type,
            f"업로드 기한 초과: 제출 {uploaded_dt}, 마감 {due_dt}",
            "마감 전까지 업로드해야 합니다."
        )
    return issues

# ---------------------------------
# 3. 거래 추출 / 매칭
# ---------------------------------
def extract_settlement_transactions(ws, target_month):
    header_row = detect_settlement_header_row(ws)
    if header_row is None:
        return pd.DataFrame()

    col_map = get_column_index_map(ws, header_row)
    evidence_cols = [k for k in col_map if str(k).startswith("증빙자료")]

    rows = []
    for row_idx in range(header_row + 1, ws.max_row + 1):
        row_dict = {k: ws.cell(row_idx, col_idx).value for k, col_idx in col_map.items()}

        core_vals = [row_dict.get("날짜"), row_dict.get("거래처명"), row_dict.get("품목"), row_dict.get("수입"), row_dict.get("지출"), row_dict.get("잔액")]
        if all(is_blank(v) for v in core_vals):
            continue

        dt = try_parse_settlement_date(row_dict.get("날짜"))
        if dt is None or dt.strftime("%Y-%m") != target_month:
            continue

        income = clean_amount(row_dict.get("수입"))
        expense = clean_amount(row_dict.get("지출"))

        if income > 0 and expense == 0:
            direction = "입금"
            abs_amount = income
        elif expense > 0 and income == 0:
            direction = "출금"
            abs_amount = expense
        else:
            direction = "혼합"
            abs_amount = abs(income - expense)

        evidence_labels = [str(row_dict.get(c)).strip() for c in evidence_cols if not is_blank(row_dict.get(c))]

        rows.append({
            "row_index": row_idx,
            "date": dt.date(),
            "datetime": dt,
            "direction": direction,
            "abs_amount": abs_amount,
            "income": income,
            "expense": expense,
            "account_title": str(row_dict.get("항") or "").strip(),
            "vendor": str(row_dict.get("거래처명") or "").strip(),
            "item": str(row_dict.get("품목") or "").strip(),
            "note": str(row_dict.get("비고") or "").strip(),
            "balance_cell": row_dict.get("잔액"),
            "evidence_labels_raw": evidence_labels,
        })

    return pd.DataFrame(rows)

def extract_bank_transactions(ws, target_month):
    header_row = find_bank_header_row(ws)
    if header_row is None:
        return pd.DataFrame()

    headers = [ws.cell(header_row, c).value for c in range(1, ws.max_column + 1)]
    col_map = {str(v).strip(): i + 1 for i, v in enumerate(headers) if v is not None}

    rows = []
    for row_idx in range(header_row + 1, ws.max_row + 1):
        raw_dt = ws.cell(row_idx, col_map["거래일시"]).value
        raw_direction = ws.cell(row_idx, col_map["구분"]).value
        raw_amount = ws.cell(row_idx, col_map["거래금액"]).value

        if all(is_blank(v) for v in [raw_dt, raw_direction, raw_amount]):
            continue

        dt = try_parse_bank_datetime(raw_dt)
        if dt is None or dt.strftime("%Y-%m") != target_month:
            continue

        rows.append({
            "row_index": row_idx,
            "date": dt.date(),
            "datetime": dt,
            "direction": "출금" if str(raw_direction).strip() == "출금" else "입금",
            "abs_amount": abs(clean_amount(raw_amount)),
            "bank_type": str(ws.cell(row_idx, col_map["거래구분"]).value or "").strip(),
            "content": str(ws.cell(row_idx, col_map["내용"]).value or "").strip(),
        })

    return pd.DataFrame(rows)

def normalize_text_for_match(text):
    text = str(text or "").strip().lower()
    text = text.replace(" ", "")
    text = text.replace("(주)", "").replace("주식회사", "")
    return text

def text_similarity(a, b):
    a = normalize_text_for_match(a)
    b = normalize_text_for_match(b)
    if a == b:
        return 1.0
    if a in b or b in a:
        return 0.95
    return SequenceMatcher(None, a, b).ratio()

def match_transactions(settlement_df, bank_df):
    matched_rows = []
    unmatched_bank = bank_df.copy()
    unmatched_settlement = []

    settlement_df = settlement_df.sort_values(["date", "direction", "abs_amount", "row_index"]).copy()
    unmatched_bank = unmatched_bank.sort_values(["date", "direction", "abs_amount", "datetime", "row_index"]).copy()

    for _, srow in settlement_df.iterrows():
        candidates = unmatched_bank[
            (unmatched_bank["date"] == srow["date"]) &
            (unmatched_bank["direction"] == srow["direction"]) &
            (unmatched_bank["abs_amount"] == srow["abs_amount"])
        ].copy()

        if candidates.empty:
            unmatched_settlement.append(srow.to_dict())
            continue

        candidates["name_similarity"] = candidates["content"].apply(lambda x: text_similarity(srow["vendor"], x))
        chosen = candidates.sort_values(["name_similarity", "datetime", "row_index"], ascending=[False, True, True]).iloc[0]

        merged = srow.to_dict()
        merged["bank_row_index"] = int(chosen["row_index"])
        merged["bank_type"] = chosen["bank_type"]
        merged["bank_content"] = chosen["content"]
        merged["name_similarity"] = float(chosen["name_similarity"])
        matched_rows.append(merged)

        unmatched_bank = unmatched_bank[unmatched_bank["row_index"] != chosen["row_index"]].copy()

    return pd.DataFrame(matched_rows), pd.DataFrame(unmatched_settlement), unmatched_bank

# ---------------------------------
# 4. 증빙 규칙
# ---------------------------------
def classify_expected_rule(row):
    text = " ".join([
        str(row.get("account_title", "")),
        str(row.get("vendor", "")),
        str(row.get("item", "")),
        str(row.get("note", "")),
        str(row.get("bank_type", "")),
        str(row.get("bank_content", "")),
    ]).lower()

    if row["direction"] == "입금":
        if "전기이월금" in text:
            return {"rule_name": "전기이월금", "required_sets": [{"전기이월금"}]}
        if "전월이월금" in text:
            return {"rule_name": "전월이월금", "required_sets": [{"전월이월금"}]}
        return {"rule_name": "입금", "required_sets": [{"입금증빙"}]}

    if any(key in text for key in ["택시", "교통", "주유", "렌터카"]):
        return {"rule_name": "교통비", "required_sets": [{"정상영수증", "사유서"}]}

    if any(key in text for key in ["미리캔버스", "어도비", "typeform", "타입폼", "툴"]):
        return {"rule_name": "툴정기결제", "required_sets": [{"카드전표", "추가증빙자료"}]}

    if "기프티콘" in text:
        return {
            "rule_name": "기프티콘",
            "required_sets": [
                {"거래명세표", "이체확인증", "상품수령증"},
                {"추가증빙자료", "이체확인증", "상품수령증"},
            ]
        }

    if "페이백" in text:
        return {
            "rule_name": "페이백",
            "required_sets": [
                {"이체확인증", "현금수령증", "정상영수증"},
                {"이체확인증", "현금수령증", "카드전표", "거래명세표"},
                {"이체확인증", "현금수령증", "간이영수증"},
            ]
        }

    if "상금" in text:
        return {"rule_name": "상금", "required_sets": [{"이체확인증", "상금수령증"}]}

    if ("상품" in str(row.get("account_title", "")) and "구매" in str(row.get("account_title", ""))) or \
       ("상품" in str(row.get("item", "")) and "구매" in str(row.get("item", ""))):
        if row.get("bank_type") == "체크카드결제":
            return {
                "rule_name": "상품구매(카드)",
                "required_sets": [{"정상영수증"}, {"카드전표", "거래명세표"}],
                "manual_review": "상품 배부용이면 상품수령증/단체상품수령증 추가 여부를 수기 검토"
            }
        return {
            "rule_name": "상품구매(이체)",
            "required_sets": [{"이체확인증", "간이영수증"}],
            "manual_review": "상품 배부용이면 상품수령증/단체상품수령증 추가 여부를 수기 검토"
        }

    if row.get("bank_type") == "체크카드결제":
        return {"rule_name": "카드결제", "required_sets": [{"정상영수증"}, {"카드전표", "거래명세표"}]}

    if row.get("bank_type") in {"일반이체", "오픈뱅킹", "자동이체(기타)"}:
        vendor = str(row.get("vendor", "")).strip()
        note = str(row.get("note", "")).strip()

        if "쿠팡" in text:
            return {"rule_name": "쿠팡이체", "required_sets": [{"이체확인증", "간이영수증"}]}

        if "학생회 간 거래" in note:
            return {"rule_name": "학생회간거래", "required_sets": [{"이체확인증"}]}

        if vendor in PERSON_LIKE_VENDORS:
            return {"rule_name": "개인거래", "required_sets": [{"이체확인증", "현금수령증"}]}

        return {"rule_name": "업체이체", "required_sets": [{"이체확인증", "간이영수증"}]}

    return {"rule_name": "기타", "required_sets": []}

def evaluate_row_rules(row):
    issues = []
    raw_labels = row["evidence_labels_raw"]
    canon_labels = [normalize_evidence_label(x) for x in raw_labels]

    # 1) 증빙자료명 표준화
    for label in raw_labels:
        base, num = split_label_number(label)
        if base not in STANDARD_EVIDENCE_LABELS:
            mapped = normalize_evidence_label(label)
            if mapped in STANDARD_EVIDENCE_LABELS:
                add_issue(
                    issues, row["row_index"], "양식 미준수",
                    f"증빙자료명 '{label}'은 표준명이 아닙니다.",
                    f"증빙자료명을 '{mapped}{num or ''}'로 수정하세요.",
                    {
                        "거래처명": row["vendor"],
                        "품목": row["item"],
                        "증빙자료": raw_labels
                    }
                )
            else:
                add_issue(
                    issues, row["row_index"], "양식 미준수",
                    f"증빙자료명 '{label}'은 교육자료 표준명에 없습니다.",
                    "표준 증빙자료명 또는 '추가증빙자료'로 수정하세요.",
                    {
                        "거래처명": row["vendor"],
                        "품목": row["item"],
                        "증빙자료": raw_labels
                    }
                )

    # 2) 거래 유형별 필수 증빙
    spec = classify_expected_rule(row)
    if spec["required_sets"]:
        passed = False
        best_missing = None

        for req_set in spec["required_sets"]:
            missing = [x for x in req_set if x not in canon_labels]
            if not missing:
                passed = True
                break
            if best_missing is None or len(missing) < len(best_missing):
                best_missing = missing

        if not passed:
            add_issue(
                issues, row["row_index"], "증빙자료 누락",
                f"{spec['rule_name']} 규칙상 필수 증빙 누락: {', '.join(best_missing)}",
                f"누락된 증빙({', '.join(best_missing)})을 추가 첨부하세요.",
                {
                    "거래처명": row["vendor"],
                    "품목": row["item"],
                    "증빙자료": raw_labels,
                    "거래구분": row.get("bank_type", "")
                }
            )

    # 3) 수기 검토 필요 항목
    if spec.get("manual_review"):
        add_issue(
            issues, row["row_index"], "검토 필요",
            spec["manual_review"],
            "메모에 삽입된 실제 이미지까지 사람이 직접 확인하세요.",
            {
                "거래처명": row["vendor"],
                "품목": row["item"],
                "증빙자료": raw_labels
            }
        )

    # 4) 거래처명 오탈자 추정
    if row.get("name_similarity", 1.0) < 0.90:
        add_issue(
            issues, row["row_index"], "검토 필요",
            f"거래처명이 통장과 상이합니다. 결산안='{row['vendor']}', 통장='{row['bank_content']}'",
            "오탈자 여부를 확인하고 결산안 거래처명을 수정하세요.",
            {
                "결산안 거래처명": row["vendor"],
                "통장 내용": row["bank_content"]
            }
        )

    # 5) 식대 반자동 규칙
    text = " ".join([str(row.get("account_title", "")), str(row.get("item", "")), str(row.get("vendor", ""))]).lower()
    if any(key in text for key in ["식대", "회식", "식사", "배달"]):
        if "식대사용보고안" not in canon_labels:
            add_issue(
                issues, row["row_index"], "검토 필요",
                "식대성 지출로 보이나 '식대사용보고안' 표기가 없습니다.",
                "실제 식대 지출이면 식대사용보고안 첨부 여부를 확인하세요.",
                {
                    "거래처명": row["vendor"],
                    "품목": row["item"],
                    "증빙자료": raw_labels
                }
            )

    return issues

def check_first_carryover_rule(settlement_df):
    issues = []
    if settlement_df.empty:
        return issues

    first_row = settlement_df.sort_values("row_index").iloc[0]
    canon_labels = [normalize_evidence_label(x) for x in first_row["evidence_labels_raw"]]
    account_title = str(first_row["account_title"])

    if "전월이월금" in account_title and "전월이월금" not in canon_labels:
        add_issue(
            issues, first_row["row_index"], "증빙자료 누락",
            "전월이월금 행에 '전월이월금' 증빙자료가 없습니다.",
            "전월이월금 캡처를 첨부하고 증빙자료명을 '전월이월금'으로 입력하세요.",
            {
                "거래처명": first_row["vendor"],
                "품목": first_row["item"],
                "증빙자료": first_row["evidence_labels_raw"]
            }
        )

    if "전기이월금" in account_title and "전기이월금" not in canon_labels:
        add_issue(
            issues, first_row["row_index"], "증빙자료 누락",
            "전기이월금 행에 '전기이월금' 증빙자료가 없습니다.",
            "전기이월금 관련 증빙을 첨부하고 증빙자료명을 '전기이월금'으로 입력하세요.",
            {
                "거래처명": first_row["vendor"],
                "품목": first_row["item"],
                "증빙자료": first_row["evidence_labels_raw"]
            }
        )

    return issues

# ---------------------------------
# 5. 전체 실행
# ---------------------------------
def run_full_audit(settlement_file, bank_file, target_month, homepage_uploaded_at=None):
    issues = []

    # 파일명 규칙
    issues.extend(check_filename_rules(settlement_file, bank_file))

    # 파일 로드
    settlement_wb, settlement_err = safe_load_workbook(settlement_file, data_only=False)
    bank_wb, bank_err = safe_load_workbook(bank_file, data_only=True)

    if settlement_err:
        add_issue(issues, 0, "검토 필요", f"결산안 파일을 열 수 없습니다: {settlement_err}", "파일을 다시 저장 후 제출하세요.")
        return pd.DataFrame(issues), pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    if bank_err:
        add_issue(issues, 0, "검토 필요", f"통장 파일을 열 수 없습니다: {bank_err}", "파일을 다시 저장 후 제출하세요.")
        return pd.DataFrame(issues), pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    settlement_ws = settlement_wb[settlement_wb.sheetnames[0]]
    bank_ws = bank_wb[bank_wb.sheetnames[0]]

    # 형식 규칙
    issues.extend(check_sheet_rules(settlement_ws, target_month))
    issues.extend(check_bank_rules(bank_ws, target_month))
    issues.extend(check_due_date_rule(homepage_uploaded_at, target_month))

    # 거래 추출
    settlement_df = extract_settlement_transactions(settlement_ws, target_month)
    bank_df = extract_bank_transactions(bank_ws, target_month)

    if settlement_df.empty:
        add_issue(issues, 0, "검토 필요", "결산안 거래를 추출하지 못했습니다.", "헤더/날짜 형식을 확인하세요.")
        return pd.DataFrame(issues), pd.DataFrame(), settlement_df, bank_df

    if bank_df.empty:
        add_issue(issues, 0, "검토 필요", "통장 거래를 추출하지 못했습니다.", "헤더/날짜 형식을 확인하세요.")
        return pd.DataFrame(issues), pd.DataFrame(), settlement_df, bank_df

    # 이월금 첫 행
    issues.extend(check_first_carryover_rule(settlement_df))

    # 거래 매칭
    matched_df, unmatched_settlement_df, unmatched_bank_df = match_transactions(settlement_df, bank_df)

    for _, row in unmatched_settlement_df.iterrows():
        add_issue(
            issues, row["row_index"], "검토 필요",
            "결산안 거래가 통장거래내역에서 매칭되지 않습니다.",
            "날짜/금액/거래처명/수입·지출 방향을 다시 확인하세요.",
            {
                "거래처명": row["vendor"],
                "품목": row["item"],
                "금액": row["abs_amount"]
            }
        )

    for _, row in unmatched_bank_df.iterrows():
        add_issue(
            issues, 0, "검토 필요",
            f"통장 거래가 결산안에서 누락되었을 가능성이 있습니다. 통장행 {int(row['row_index'])}",
            "누락 거래가 결산안에 반영되었는지 확인하세요.",
            {
                "통장행": int(row["row_index"]),
                "통장 내용": row["content"],
                "금액": row["abs_amount"]
            }
        )

    # 행별 증빙 규칙
    for _, row in matched_df.iterrows():
        issues.extend(evaluate_row_rules(row))

    audit_df = pd.DataFrame(issues)
    if not audit_df.empty:
        audit_df = audit_df.sort_values(["penalty_points", "row_index", "issue_type"], ascending=[False, True, True]).reset_index(drop=True)

    return audit_df, matched_df, settlement_df, bank_df


# =========================
# 실행
# =========================
audit_df, matched_df, settlement_df, bank_df = run_full_audit(
    settlement_file=settlement_file,
    bank_file=bank_file,
    target_month=target_month,
    homepage_uploaded_at=homepage_uploaded_at
)

# 1) 행별 시정사항 / 벌점
audit_df

# 2) 벌점 합계표
# audit_df.groupby("issue_type", dropna=False)["penalty_points"].sum().sort_values(ascending=False)

# 3) 거래 매칭 상세표
# matched_df

c:\Users\daeha\AppData\Local\Programs\Python\Python314\Lib\site-packages\openpyxl\worksheet\header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


,row_index,issue_type,penalty_points,issue_summary,expected_fix,row_snapshot
0,11,증빙자료 누락,5,상품구매(이체) 규칙상 필수 증빙 누락: 간이영수증,누락된 증빙(간이영수증)을 추가 첨부하세요.,"{'거래처명': '쿠팡', '품목': '신입생 OT 이벤트 상품 구매', '증빙자료..."
1,15,증빙자료 누락,5,업체이체 규칙상 필수 증빙 누락: 간이영수증,누락된 증빙(간이영수증)을 추가 첨부하세요.,"{'거래처명': '아바타펜션', '품목': 'MT 숙소 예약금', '증빙자료': [..."
2,17,증빙자료 누락,5,상품구매(이체) 규칙상 필수 증빙 누락: 간이영수증,누락된 증빙(간이영수증)을 추가 첨부하세요.,"{'거래처명': '쿠팡', '품목': '음악과 팔로우 이벤트 상품 구매', '증빙자..."
3,22,증빙자료 누락,5,업체이체 규칙상 필수 증빙 누락: 간이영수증,누락된 증빙(간이영수증)을 추가 첨부하세요.,"{'거래처명': '정광열', '품목': 'MT 버스 대절비', '증빙자료': ['이..."
4,28,증빙자료 누락,5,상품구매(이체) 규칙상 필수 증빙 누락: 간이영수증,누락된 증빙(간이영수증)을 추가 첨부하세요.,"{'거래처명': '쿠팡', '품목': '신입생 OT 이벤트 상품 구매', '증빙자료..."
5,42,증빙자료 누락,5,쿠팡이체 규칙상 필수 증빙 누락: 간이영수증,누락된 증빙(간이영수증)을 추가 첨부하세요.,"{'거래처명': '쿠팡', '품목': '학생회 이벤트 필요물품 구매', '증빙자료'..."
6,52,증빙자료 누락,5,상품구매(이체) 규칙상 필수 증빙 누락: 간이영수증,누락된 증빙(간이영수증)을 추가 첨부하세요.,"{'거래처명': '쿠팡이츠', '품목': '샐러리아 아이스티 20잔', '증빙자료'..."
7,87,증빙자료 누락,5,업체이체 규칙상 필수 증빙 누락: 간이영수증,누락된 증빙(간이영수증)을 추가 첨부하세요.,"{'거래처명': '㈜더아이핏', '품목': '음악과 MT 과티 구매', '증빙자료'..."
8,88,증빙자료 누락,5,개인거래 규칙상 필수 증빙 누락: 현금수령증,누락된 증빙(현금수령증)을 추가 첨부하세요.,"{'거래처명': '트래볼로버', '품목': '음악과 MT 여행자 보험 입금', '증..."
9,89,증빙자료 누락,5,업체이체 규칙상 필수 증빙 누락: 간이영수증,누락된 증빙(간이영수증)을 추가 첨부하세요.,"{'거래처명': '김은혜', '품목': '음악과 MT 참가비 환불', '증빙자료':..."
